# Customer Churn Prediction - Telecom

**Tabular Classification** — Predict customer churn for a telecom company.

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Dataset: Telco Customer Churn (7,043 rows × 21 columns)
Target: `Churn`

## 2 · Project Overview

We predict whether a telecom customer will **churn** (leave the service) based on their account information, demographics, and subscribed services. Customer retention is far cheaper than acquisition, making churn prediction a high-value business application.

## 3 · Learning Objectives

1. Build an end-to-end churn prediction pipeline.
2. Handle mixed data types (numeric + categorical).
3. Engineer customer value and engagement features.
4. Compare boosting models for churn classification.
5. Understand business implications of churn prediction.

## 4 · Problem Statement

Given a customer's **demographics** (gender, senior citizen, partner, dependents), **account info** (tenure, contract, billing, charges), and **services** (phone, internet, streaming, security), predict whether they will **churn**.

## 5 · Why This Project Matters

- Customer acquisition costs 5–25x more than retention.
- Identifying at-risk customers enables proactive retention offers.
- Telecom churn rates are typically 15–25% annually.
- This is one of the most common ML business use cases across industries.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,043 |
| **Columns** | 21 |
| **Features** | customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges |
| **Target** | `Churn` (Yes/No) |
| **Source** | kagglehub: `blastchar/telco-customer-churn` |

## 7 · Dataset Source and License Notes

- **Source**: IBM Sample Datasets, distributed on Kaggle.
- **Kaggle**: `blastchar/telco-customer-churn`
- **License**: Public / Educational use.
- **Note**: Widely used in industry workshops and ML courses.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [("catboost", None), ("lightgbm", None), ("xgboost", None),
                 ("flaml", None), ("lazypredict", None), ("kagglehub", None)]:
    _install_if_missing(pkg, imp)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, re, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, average_precision_score)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("Imports done.")

Imports done.


## 10 · Configuration

In [3]:
SAVE_DIR = r"E:/Github/Machine-Learning-Projects/Classification/Customer Churn Prediction - Telecom"
os.makedirs(SAVE_DIR, exist_ok=True)
TARGET = "Churn"
print(f"Save dir: {SAVE_DIR}")

Save dir: E:/Github/Machine-Learning-Projects/Classification/Customer Churn Prediction - Telecom


## 11 · Dataset Download and Loading

In [4]:
import kagglehub
data_path = kagglehub.dataset_download("blastchar/telco-customer-churn")
import glob
csv_file = glob.glob(os.path.join(data_path, "*.csv"))[0]
df = pd.read_csv(csv_file)

# Drop customerID — not a feature
if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

# Fix TotalCharges — has whitespace values
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.head()

Shape: (7043, 20)
Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 12 · Data Validation Checks

In [5]:
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nTarget unique: {df[TARGET].unique()}")
print(f"\nCategorical columns: {df.select_dtypes(include=['object']).columns.tolist()}")

Missing values:
Series([], dtype: int64)
No missing values.

Duplicates: 22

Target unique: ['No' 'Yes']

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["tenure"].hist(bins=30, ax=axes[0], edgecolor="black")
axes[0].set_title("Tenure (months)")
df["MonthlyCharges"].hist(bins=30, ax=axes[1], edgecolor="black", color="orange")
axes[1].set_title("Monthly Charges")
df["TotalCharges"].hist(bins=30, ax=axes[2], edgecolor="black", color="green")
axes[2].set_title("Total Charges")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

# Churn rate by contract type
if "Contract" in df.columns:
    ct = df.groupby("Contract")[TARGET].apply(lambda x: (x == "Yes").mean())
    print(f"\nChurn rate by Contract:\n{ct}")


Churn rate by Contract:
Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64


## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Churn Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_target.png"), dpi=100)
plt.show()
print(f"Churn rate: {(df[TARGET] == 'Yes').mean():.4f}")

Churn rate: 0.2654


## 15 · Train / Test Split

In [8]:
# Encode target
df[TARGET] = (df[TARGET] == "Yes").astype(int)

# Encode all categorical features
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

y = df[TARGET]
X = df.drop(columns=[TARGET])

# Sanitize column names
X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.4f}, test: {y_test.mean():.4f}")

Train: (5634, 19), Test: (1409, 19)
Churn rate — train: 0.2654, test: 0.2654


## 16 · Preprocessing Strategy

- `TotalCharges` whitespace converted to numeric with median imputation.
- All categorical features label-encoded.
- `customerID` dropped (identifier, not a feature).
- No scaling needed for tree-based models.

In [9]:
print(f"Missing after prep: {X_train.isnull().sum().sum()}")
print(f"All features numeric: {X_train.shape[1]}")

Missing after prep: 0
All features numeric: 19


## 17 · Feature Engineering

In [10]:
def add_features(df_):
    df_ = df_.copy()
    if "tenure" in df_.columns and "MonthlyCharges" in df_.columns:
        df_["avg_monthly_spend"] = df_["TotalCharges"] / (df_["tenure"] + 1)
        df_["charge_ratio"] = df_["MonthlyCharges"] / (df_["TotalCharges"] + 1)
    return df_

X_train = add_features(X_train)
X_test = add_features(X_test)
print(f"Features after engineering: {X_train.shape[1]}")

Features after engineering: 21


## 18 · Baseline Model

In [11]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)
print(f"Baseline Logistic Regression:")
print(f"  Accuracy: {baseline_acc:.4f}  F1: {baseline_f1:.4f}  ROC-AUC: {baseline_auc:.4f}")
print(classification_report(y_test, baseline_pred))

Baseline Logistic Regression:
  Accuracy: 0.7452  F1: 0.6217  ROC-AUC: 0.8477
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.75      0.76      1409



## 19 · LazyPredict Benchmark

In [12]:
# LazyPredict — quick benchmark of many classifiers
from lazypredict.Supervised import LazyClassifier

lp_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
lp_models, lp_predictions = lp_clf.fit(X_train, X_test, y_train, y_test)

print("\nLazyPredict Results (top 15):")
print(lp_models.head(15).to_string())


LazyPredict Results (top 15):
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
GaussianNB                     0.738822           0.746233  0.828601  0.752076   0.791325  0.738822    0.018684
NearestCentroid                0.719659           0.744289  0.822806  0.735387   0.792276  0.719659    0.021189
BernoulliNB                    0.750177           0.736887  0.821317  0.760509   0.784164  0.750177    0.020661
QuadraticDiscriminantAnalysis  0.679205           0.735537  0.840019  0.697765   0.795460  0.679205    0.024763
AdaBoostClassifier             0.801278           0.724710  0.843889  0.796716   0.794314  0.801278    0.274066
LinearDiscriminantAnalysis     0.802697           0.723969  0.843173  0.797529   0.795125  0.802697    0.023745
LogisticRegression             0.804826           0.718587  0.848157  0.7

## 20 · FLAML AutoML

In [13]:
# FLAML AutoML
try:
    from flaml import AutoML
    flaml_clf = AutoML()
    flaml_clf.fit(X_train, y_train, task="classification", time_budget=60,
                  metric="f1", verbose=0, seed=SEED)
    flaml_pred = flaml_clf.predict(X_test)
    flaml_proba = flaml_clf.predict_proba(X_test)[:, 1] if hasattr(flaml_clf, "predict_proba") else None
    print(f"FLAML best model: {flaml_clf.best_estimator}")
    print(f"FLAML F1: {f1_score(y_test, flaml_pred):.4f}")
    print(f"FLAML Accuracy: {accuracy_score(y_test, flaml_pred):.4f}")
    if flaml_proba is not None:
        print(f"FLAML ROC-AUC: {roc_auc_score(y_test, flaml_proba):.4f}")
    flaml_ok = True
except Exception as e:
    print(f"FLAML failed: {e}")
    flaml_ok = False

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Top 3 Models — CatBoost, LightGBM, XGBoost

In [14]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import torch
device_cat = "GPU" if torch.cuda.is_available() else "CPU"
device_lgb = "gpu" if torch.cuda.is_available() else "cpu"
device_xgb = "cuda" if torch.cuda.is_available() else "cpu"

models = {
    "CatBoost": CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, verbose=0,
        task_type=device_cat, random_seed=SEED, eval_metric="F1",
        early_stopping_rounds=50
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_lgb, random_state=SEED, verbose=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_xgb, random_state=SEED, eval_metric="logloss",
        early_stopping_rounds=50
    ),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    t0 = time.time()
    if name == "XGBoost":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == "CatBoost":
        model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)
    else:
        model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, proba)

    results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "time": elapsed}
    print(f"  Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}  ({elapsed:.1f}s)")
    print(classification_report(y_test, preds))

results_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\n=== Top 3 Model Comparison ===")
print(results_df.to_string())


Training CatBoost...


  Accuracy: 0.8105  Precision: 0.6918  Recall: 0.5160  F1: 0.5911  ROC-AUC: 0.8471  (8.8s)
              precision    recall  f1-score   support

           0       0.84      0.92      0.88      1035
           1       0.69      0.52      0.59       374

    accuracy                           0.81      1409
   macro avg       0.77      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409


Training LightGBM...


  Accuracy: 0.7906  Precision: 0.6254  Recall: 0.5267  F1: 0.5718  ROC-AUC: 0.8314  (2.1s)
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.78      1409


Training XGBoost...


  Accuracy: 0.7991  Precision: 0.6564  Recall: 0.5107  F1: 0.5744  ROC-AUC: 0.8399  (0.5s)
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409


=== Top 3 Model Comparison ===
          accuracy  precision    recall        f1   roc_auc      time
CatBoost  0.810504   0.691756  0.516043  0.591118  0.847079  8.785078
XGBoost   0.799148   0.656357  0.510695  0.574436  0.839930  0.474433
LightGBM  0.790632   0.625397  0.526738  0.571843  0.831357  2.109610


## 22 · Top 3 Model Selection

In [15]:
print("=== Final Ranking ===")
print(results_df[["f1", "roc_auc", "accuracy", "precision", "recall"]].to_string())

=== Final Ranking ===
                f1   roc_auc  accuracy  precision    recall
CatBoost  0.591118  0.847079  0.810504   0.691756  0.516043
XGBoost   0.574436  0.839930  0.799148   0.656357  0.510695
LightGBM  0.571843  0.831357  0.790632   0.625397  0.526738


## 23 · Final Evaluation

In [16]:
print("Comparison vs baseline:")
print(f"  Baseline F1: {baseline_f1:.4f} | ROC-AUC: {baseline_auc:.4f}")
for name, vals in results.items():
    improvement = (vals["f1"] - baseline_f1) / max(baseline_f1, 0.001) * 100
    print(f"  {name} F1: {vals['f1']:.4f} | ROC-AUC: {vals['roc_auc']:.4f} | F1 improvement: {improvement:+.1f}%")

Comparison vs baseline:
  Baseline F1: 0.6217 | ROC-AUC: 0.8477
  CatBoost F1: 0.5911 | ROC-AUC: 0.8471 | F1 improvement: -4.9%
  LightGBM F1: 0.5718 | ROC-AUC: 0.8314 | F1 improvement: -8.0%
  XGBoost F1: 0.5744 | ROC-AUC: 0.8399 | F1 improvement: -7.6%


## 24 · Error Analysis

In [17]:
# Error Analysis — examine misclassifications from best model
best_name = results_df.index[0]
best_model = models[best_name]
best_preds = best_model.predict(X_test)

errors = X_test.copy()
errors["y_true"] = y_test.values
errors["y_pred"] = best_preds
errors["correct"] = errors["y_true"] == errors["y_pred"]

print(f"Best model: {best_name}")
print(f"Total test samples: {len(errors)}")
print(f"Correct: {errors['correct'].sum()} | Wrong: {(~errors['correct']).sum()}")
print(f"Error rate: {(~errors['correct']).mean():.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()
print("Confusion matrix saved.")

Best model: CatBoost
Total test samples: 1409
Correct: 1142 | Wrong: 267
Error rate: 0.1895
Confusion matrix saved.


## 25 · Interpretation and Business Insight

- **Tenure** is the strongest predictor — new customers are most likely to churn.
- **Contract type** matters hugely — month-to-month customers churn 3–5x more.
- **Monthly charges** correlate with churn — higher bills drive attrition.
- Customers without tech support and online security churn more.

## 26 · Limitations

- Single telecom company — patterns may not generalize.
- No customer interaction data (calls to support, complaints).
- No competitive info (better offers from competitors).
- Static snapshot — no temporal churn dynamics.

## 27 · How to Improve

1. Add customer interaction features (support calls, complaints).
2. Use survival analysis for time-to-churn modeling.
3. Implement uplift modeling to identify customers who respond to interventions.
4. Build customer lifetime value (CLV) predictions alongside churn.

## 28 · Production Considerations

- Need real-time scoring as customer behavior changes.
- Integration with CRM for automated retention campaigns.
- A/B testing of retention offers triggered by churn predictions.
- Monitor churn rate trends and model drift over time.

## 29 · Common Mistakes

1. Not accounting for class imbalance (~26% churn).
2. Using accuracy instead of F1/ROC-AUC.
3. Not removing customerID before training.
4. Ignoring TotalCharges whitespace/missing values.

## 30 · Mini Challenge

1. Calculate customer lifetime value and add it as a feature.
2. Build a retention offer recommendation system based on churn probability.
3. Compare SMOTE vs class weights for handling imbalance.
4. Create customer segments based on churn risk (low/medium/high).

## 31 · Final Summary

- Churn prediction is one of the highest-ROI ML applications in telecom.
- Tenure, contract type, and monthly charges drive churn most.
- Gradient boosting models provide strong baseline performance.
- Business value comes from combining predictions with retention actions.

In [18]:
# Save metrics to JSON
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in results.items()}
metrics["baseline"] = {"accuracy": round(baseline_acc, 4), "f1": round(baseline_f1, 4)}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {os.path.join(SAVE_DIR, 'metrics.json')}")

Metrics saved to E:/Github/Machine-Learning-Projects/Classification/Customer Churn Prediction - Telecom\metrics.json
